In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Set plot styling for clean, professional visuals
sns.set_theme(style="whitegrid")

In [4]:
# 1. Load the original dataset
file_path = "BLC-10.csv"
df = pd.read_csv(file_path)

# 2. Standardize categorical text
df["Category"] = df["Category"].str.strip().str.title().fillna("Unknown")

# 3. Categorization logic mapping dataset items to your 13 specified fields
def map_to_standard_category(cat):
    cat_lower = str(cat).lower().strip()
    
    if 'coaching' in cat_lower or 'caoching' in cat_lower:
        return 'Coaching Center'
    if 'tutoring' in cat_lower or 'tuition' in cat_lower:
        return 'Academic Tuition'
    if 'training' in cat_lower or 'education' in cat_lower or 'educational' in cat_lower or 'institute' in cat_lower or 'university' in cat_lower or 'school' in cat_lower or 'computer center' in cat_lower or 'vocational' in cat_lower:
        return 'Training Center'
    if 'dermatologist' in cat_lower or 'skin' in cat_lower:
        return 'Dermatologist'
    if 'dental' in cat_lower or 'dentist' in cat_lower:
        return 'Dental Clinic'
    if 'physio' in cat_lower:
        return 'Physiotherapy Clinic'
    if 'pharmacy' in cat_lower:
        return 'Pharmacy'
    if 'restaurant' in cat_lower or 'food' in cat_lower:
        return 'Restaurant'
    if 'supermarket' in cat_lower:
        return 'Supermarket'
    if 'bakery' in cat_lower:
        return 'Bakery'
    if 'salon' in cat_lower or 'spa' in cat_lower or 'gym' in cat_lower:
        return 'Salon'
    if 'grocery' in cat_lower:
        return 'Grocery Store'
    if 'dairy' in cat_lower:
        return 'Dairy Shop'
        
    # High-intent medical/clinical fallback grouping 
    if any(keyword in cat_lower for keyword in ['medical', 'clinic', 'hospital', 'doctor', 'homeopath', 'health', 'surgeon', 'pediatrician', 'practitioner', 'care', 'maternity', 'nursing', 'acupuncture', 'ayurvedic', 'holistic']):
        return 'Physiotherapy Clinic'
        
    return 'Training Center'

df['Standard_Category'] = df['Category'].apply(map_to_standard_category)
print(f"Data Loaded and Categorized. Total records: {len(df)}")

Data Loaded and Categorized. Total records: 292


In [5]:
# Define Priority Ranks from your guidelines
india_ranks = {
    'Training Center': 1, 'Coaching Center': 2, 'Dermatologist': 3, 'Dental Clinic': 4,
    'Physiotherapy Clinic': 5, 'Restaurant': 6, 'Supermarket': 7, 'Academic Tuition': 8,
    'Bakery': 9, 'Salon': 10, 'Pharmacy': 11, 'Grocery Store': 12, 'Dairy Shop': 13
}

foreign_ranks = {
    'Dental Clinic': 1, 'Dermatologist': 2, 'Physiotherapy Clinic': 3, 'Training Center': 4,
    'Salon': 5, 'Restaurant': 6, 'Bakery': 7, 'Academic Tuition': 8, 'Coaching Center': 9,
    'Pharmacy': 10, 'Supermarket': 11, 'Grocery Store': 12, 'Dairy Shop': 13
}

# Apply regional ground-truth scoring mappings
def assign_lead_score_and_rank(row):
    country = str(row['Country']).strip().lower()
    cat = row['Standard_Category']
    
    if 'india' in country:
        rank = india_ranks.get(cat, 13)
        if rank <= 5: return pd.Series([0.85, rank])      # High Priority Block
        elif rank <= 10: return pd.Series([0.55, rank])    # Medium Priority Block
        else: return pd.Series([0.15, rank])               # Low Priority Block
    else:
        rank = foreign_ranks.get(cat, 13)
        if rank <= 3: return pd.Series([0.85, rank])      # High-Ticket Medical Block
        elif rank <= 7: return pd.Series([0.55, rank])    # Premium Lifestyle/Food Block
        else: return pd.Series([0.15, rank])               # Low/Consolidated Retail Block

df[['Conversion_Probability', 'Rank']] = df.apply(assign_lead_score_and_rank, axis=1)
print(df[['Country', 'Standard_Category', 'Rank', 'Conversion_Probability']].head())

  Country Standard_Category  Rank  Conversion_Probability
0   India          Pharmacy  11.0                    0.15
1   India          Pharmacy  11.0                    0.15
2   India          Pharmacy  11.0                    0.15
3   India          Pharmacy  11.0                    0.15
4   India          Pharmacy  11.0                    0.15


In [6]:
print("Generating Exploratory Data Analysis & Plotting Graphs...")

# --- GRAPH 1: Distribution of Businesses by City ---
city_data = df["City"].value_counts().reset_index()
city_data.columns = ["City", "Count"]
city_data = city_data.sort_values(by="Count", ascending=False).head(15)

fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.barplot(data=city_data, x="Count", y="City", palette="Blues_r", ax=ax1)
ax1.set_title("Distribution of Businesses by City (Top 15)", fontsize=14, fontweight="bold")
ax1.set_xlabel("Number of Businesses", fontsize=12)
ax1.set_ylabel("City", fontsize=12)
plt.tight_layout()
plt.savefig("distribution_of_businesses_by_city.png")
plt.close()

# --- GRAPH 2: Business Categories by Total Count ---
category_counts = df["Standard_Category"].value_counts().reset_index()
category_counts.columns = ["Standard_Category", "Count"]
category_counts = category_counts.sort_values(by="Count", ascending=False)

fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="mako", ax=ax2)
ax2.set_title("Business Categories by Total Count", fontsize=14, fontweight="bold")
ax2.set_xlabel("Number of Businesses", fontsize=12)
ax2.set_ylabel("Business Category", fontsize=12)
plt.tight_layout()
plt.savefig("top_business_categories_by_count.png")
plt.close()

print("Initial Graphs Generated successfully.")

Generating Exploratory Data Analysis & Plotting Graphs...
Initial Graphs Generated successfully.


/var/folders/0d/c1m6ktfs6yzf2kk_1jhnj2g00000gn/T/ipykernel_8279/963016871.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=city_data, x="Count", y="City", palette="Blues_r", ax=ax1)
/var/folders/0d/c1m6ktfs6yzf2kk_1jhnj2g00000gn/T/ipykernel_8279/963016871.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="mako", ax=ax2)


In [7]:
# Structural features from text strings
df["Company_Name_Length"] = df["Company"].str.len()
df["Company_Word_Count"] = df["Company"].str.split().str.len()

# Saturated metrics / Frequency weight maps
category_counts_dict = df["Standard_Category"].value_counts().to_dict()
df["Category_Frequency"] = df["Standard_Category"].map(category_counts_dict)

city_counts_dict = df["City"].value_counts().to_dict()
df["City_Frequency"] = df["City"].map(city_counts_dict)

# Label encoding for categoricals
le_city = LabelEncoder()
df["City_Encoded"] = le_city.fit_transform(df["City"])

le_category = LabelEncoder()
df["Category_Encoded"] = le_category.fit_transform(df["Standard_Category"])

print("Engineered Features Preview:")
print(df[["Company_Name_Length", "Company_Word_Count", "Category_Frequency", "City_Encoded"]].head())

Engineered Features Preview:
   Company_Name_Length  Company_Word_Count  Category_Frequency  City_Encoded
0                   23                   4                  20             0
1                   12                   3                  20             0
2                   11                   2                  20             0
3                   13                   2                  20             0
4                   26                   4                  20             0


In [8]:
# Define Features and Target
feature_names = ["Company_Name_Length", "Company_Word_Count", "Category_Frequency", "City_Frequency", "City_Encoded", "Category_Encoded"]
X = df[feature_names]
y = df["Conversion_Probability"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Regressor Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print(f"Model Training R^2 Score: {rf_model.score(X_train, y_train):.4f}")
print(f"Model Testing R^2 Score: {rf_model.score(X_test, y_test):.4f}")

Model Training R^2 Score: 0.9998
Model Testing R^2 Score: 1.0000


In [9]:
# Run predictions across all rows
df["Predicted_Probability"] = rf_model.predict(X)
df["Google Maps"] = df["Google Map"] # Ensure column naming matches configuration expectations

# --- GRAPH 3: Business Categories by Predicted Conversion Probability ---
category_rankings = df.groupby("Standard_Category")["Predicted_Probability"].mean().reset_index()
category_rankings = category_rankings.sort_values(by="Predicted_Probability", ascending=False)

fig3, ax3 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)
ax3.set_title("Business Categories by Conversion Probability", fontsize=14, fontweight="bold")
ax3.set_xlabel("Average Predicted Conversion Probability", fontsize=12)
ax3.set_ylabel("Business Category", fontsize=12)
plt.tight_layout()
plt.savefig("top_business_categories_by_probability.png")
plt.close()

# --- GRAPH 4: Lead Volume Breakdown by Country ---
country_counts = df["Country"].value_counts().reset_index()
country_counts.columns = ["Country", "Count"]

fig4, ax4 = plt.subplots(figsize=(8, 5))
sns.barplot(data=country_counts, x="Count", y="Country", palette="coolwarm", ax=ax4)
ax4.set_title("Lead Volume by Country", fontsize=14, fontweight="bold")
ax4.set_xlabel("Number of Businesses", fontsize=12)
ax4.set_ylabel("Country", fontsize=12)
plt.tight_layout()
plt.savefig("lead_volume_by_country.png")
plt.close()

print("All analytics graphs saved successfully.")

/var/folders/0d/c1m6ktfs6yzf2kk_1jhnj2g00000gn/T/ipykernel_8279/2539420547.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)
/var/folders/0d/c1m6ktfs6yzf2kk_1jhnj2g00000gn/T/ipykernel_8279/2539420547.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=country_counts, x="Count", y="Country", palette="coolwarm", ax=ax4)


All analytics graphs saved successfully.


In [12]:
# Filter the necessary final columns for clean visibility
columns_to_export = ["S.No", "Country", "City", "Company", "CONTACT", "Google Maps", "Category", "Standard_Category", "Rank", "Predicted_Probability"]

# Segregate into separate DataFrames and sort ascending by Rank (Rank 1 sits at the very top)
indian_df = df[df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]
foreign_df = df[~df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]

# Initialize openpyxl Engine
wb = openpyxl.Workbook()
wb.remove(wb.active)  # Strip default worksheet

# Set configuration maps (Tab Name -> DataFrame, Header Fill Color Hex)
sheets_config = {
    f"Indian Businesses ({len(indian_df)})": (indian_df, "365F91"),   # Corporate Blue
    f"Foreign Businesses ({len(foreign_df)})": (foreign_df, "5C5C5C") # Slate Gray
}

for sheet_name, (sub_df, header_color) in sheets_config.items():
    ws = wb.create_sheet(title=sheet_name)
    ws.views.sheetView[0].showGridLines = True  # Explicitly force Excel grid lines to show
    
    headers = list(sub_df.columns)
    ws.append(headers)
    
    # Header & Row Styles
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color=header_color, end_color=header_color, fill_type="solid")
    center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center")
    thin_border = Border(
        left=Side(style="thin", color="DDDDDD"), right=Side(style="thin", color="DDDDDD"),
        top=Side(style="thin", color="DDDDDD"), bottom=Side(style="thin", color="DDDDDD")
    )
    
    # Format Headers
    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center_align
        
    # Format and Populate Rows
    for r_idx, row_data in enumerate(sub_df.values, 2):
        for c_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            cell.border = thin_border
            cell.font = Font(name="Calibri", size=11)
            
            current_header = headers[c_idx - 1]
            
            # Context Alignment and Number Formatting Layout rules
            if current_header in ["S.No", "CONTACT", "Rank"]:
                cell.alignment = center_align
                if current_header == "CONTACT":
                    cell.number_format = "@"  # Text explicit formatting rules out loss of leading characters
            elif current_header == "Predicted_Probability":
                cell.alignment = center_align
                cell.number_format = "0.0%"  # Display metrics cleanly as a standard percentage
            else:
                cell.alignment = left_align
                
    # Auto-adjust column dynamic spacing constraints
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

# Save Outputs Safely
wb.save("leads_prioritized_by_country.xlsx")
df.to_csv("business_leads_prioritized 6.csv", index=False)

print("\n--- PROCESS COMPLETE ---")
print("Successfully generated 'business_leads_prioritized.csv'")
print("Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!")


--- PROCESS COMPLETE ---
Successfully generated 'business_leads_prioritized.csv'
Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!
